In [ ]:
!pip install PyPDF2 faiss-cpu sentence-transformers gradio 

Loading and Parsing the PDF:
To extract the text from the PDF, I wrote a function called extract_text_from_pdf. I opened the pdf file in binary mode so I could feed it directly into PdfReader, which reads the document page by page. For each page, I used extract_text() to grab the content, and I made sure to skip any empty pages to avoid cluttering the results. I also added line breaks between pages to keep the extracted text readable and logically separated which I knew would help later when chunking the data.

Chunking Strategy:
Once I had the full text, I split it into chunks of 100 characters. I chose a fixed size because embedding models like all-MiniLM-L6-v2 work better with shorter, consistent-length text. Smaller chunks also mean that when a query is made, the system can pinpoint more specific parts of the document instead of giving longer answers.

Embeddings:
I then loaded the all-MiniLM-L6-v2 model from SentenceTransformer because it was told to be a balance between speed and semantic accuracy which is suitable for lightweight application such as this one. I encoded each chunk into a vector embedding. I determined the vector dimension from the embedding shape and passed that into FAISS to make sure the index matched the model output.

Building the FAISS Index:
For the similarity search, I went with IndexFlatL2 from FAISS, which calculates exact nearest-neighbor matches using L2 distance. I chose this method because it’s simple, and works well with sentence embeddings for relevance ranking. Before adding the embeddings, I converted them to float32 format since FAISS expects that data type.

get_answer(question) Function:
When someone asks a question, I convert it into a special number code (called an embedding) using my sentence transformer model. This helps me understand the meaning behind the words. Then I compare this number code against all the text snippets from the report using my FAISS index - it's like a super-fast matching system that finds the most similar piece of text. I only fetch the single best match which is why k=1 to keep things simple. Finally, I return that raw text snippet exactly as it appears in the document.
I chose Gradio interface to build the UI where user input a question that will retrieve the text. To make testing easier, I pre-loaded the exact questions from the asssessment in the examples arrays.

Issues faced and Problems
The system often gives incomplete or unclear output because of the 100 character chunking method. Important stuff like numbers and tables get split between chunks, so when you ask about death counts or positivity rates, you often get half an answer. As a result, it may return text without enough context. This is a problem for questions needing comparisons or multi-part answers,where states and their values are not shown together even if both are in the document.
The method is also not designed for approximate search. In situations where real-time responses are needed over large datasets, other indexes such as approximate nearest neighbor(ANN) indexes offer much faster retrieval while still returning highly relevant results. 

Proposed fixes
Instead of arbitrarily cutting the report into fixed-length pieces (which often chopped sentences and data points in half), I should have split the document at natural break points like paragraph endings and section headers. This keeps complete ideas together such as entire regional case summary stays in one chunk. Tables also should be specified in the way that the system extracted each as a whole and instantly recognizes structured datasets.
There are also many others dimension model that outperforms MiniLM in semantic understanding. This helps distinguish similar medical terms like "positivity rate" vs "incidence rate" and understands numerical comparisons than MiniLm. Besides that,using IndexFlatIP (inner product) instead of L2 distance could potentially improve the system better at matching meaning than just word patterns.

In [ ]:
import PyPDF2
import numpy as np
import gradio as gr
from sentence_transformers import SentenceTransformer
import faiss
pdf_path = r"C:\Users\Al\Desktop\Questions_RND\Question3\COVID19_sitrep_MYS_w-46--47.pdf"

# 1. Parsing the document
def extract_text_from_pdf(pdf_path):
    """Extracts text from a PDF file"""
    text = ""
    with open(pdf_path, 'rb') as file:
        reader = PyPDF2.PdfReader(file)
        for page in reader.pages:
            page_text = page.extract_text()
            if page_text: 
                text += page_text + "\n"
    return text

# Extracting text from PDF file
pdf_text = extract_text_from_pdf(pdf_path)

# 2. Creating text chunks
chunk_size = 100  # Characters per chunk
chunks = [pdf_text[i:i+chunk_size] for i in range(0, len(pdf_text), chunk_size)]

# 3. Creating embeddings
model = SentenceTransformer('all-MiniLM-L6-v2')  # Lightweight embedding model
embeddings = model.encode(chunks)  # Convert text chunks to vectors

# 4. Create FAISS index
dimension = embeddings.shape[1]  # Get dimension of the embedding
index = faiss.IndexFlatL2(dimension)  # L2 distance index

# Add embeddings to index (convert to float32 for FAISS)
index.add(np.array(embeddings).astype('float32'))

# Store data for retrieval
database = {
    "index": index,
    "chunks": chunks,
    "embedding_model": model
}

In [21]:
def get_answer(question):
    # Convert question to embedding
    question_embed = model.encode([question])
    
    # Search in FAISS index 
    distances, indices = database["index"].search(question_embed, k=1)
    
    # Return best matching chunk
    return database["chunks"][indices[0][0]]

# Create  Gradio input interface together with the predefined examples in an array
iface = gr.Interface(
    fn=get_answer,
    inputs=gr.Textbox(lines=2, label="Your Question"),
    outputs=gr.Textbox(label="c Relevant Text from Report"),
    title="COVID-19 Report QA System",
    description="Ask specific questions about Malaysia's Covid-19 situation report",
    examples=[
        ["What was the total number of confirmed COVID-19 cases as of 27 November 2022?" ],
        ["What was the positivity rate in FT Putrajaya over the last 14 days? "],
        ["How many COVID-19 deaths were reported in the past two weeks? "],
        ["What was the cumulative positivity rate for COVID-19 testing since 1 January 2022? "],
        ["Which two states had the highest 14-day positivity rates, and what were the rates?"]
        
    ]
)

iface.launch(share=True)

* Running on local URL:  http://127.0.0.1:7865
* Running on public URL: https://94419022244f09962e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
